In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q08/annotated/checkpoints/pre_cell_7.pickle

trying: ['lineitem']


me:  2
trying: ['load_supplier']
me:  3
trying: ['orders']


me:  4
trying: ['DATA_ROOT']
me:  0
trying: ['nation']
me:  6
trying: ['load_customer']
me:  5
trying: ['part']
me:  1
trying: ['load_orders']
me:  4
trying: ['supplier']
me:  3
trying: ['load_part']
me:  1
trying: ['customer']
me:  5
trying: ['tpch_parent']
me:  0
trying: ['load_nation']
me:  6
trying: ['load_region']
me:  7
trying: ['region']
me:  7
trying: ['load_lineitem']
me:  2
trying: ['STORAGE_OPTS']
me:  0
trying: ['pd']
me:  0


Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable STORAGE_OPTS
Declaring variable pd
Declaring variable part
Declaring variable load_part
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable load_supplier
Declaring variable supplier
Declaring variable orders
Declaring variable load_orders
Declaring variable load_customer
Declaring variable customer
Declaring variable nation
Declaring variable load_nation
Declaring variable load_region
Declaring variable region


In [4]:
%%RecordEvent
%%time
### cell 7 ###

### cell 7 (optimized for cudf)

# 1) filter part and project key
part_filtered = part[part.P_TYPE == "ECONOMY ANODIZED STEEL"][["P_PARTKEY"]]

# 2) compute volume on lineitem and project
lineitem_filtered = lineitem[["L_PARTKEY", "L_SUPPKEY", "L_ORDERKEY"]]
lineitem_filtered["VOLUME"] = (
    lineitem.L_EXTENDEDPRICE * (1.0 - lineitem.L_DISCOUNT)
)

# 3) join part → lineitem
total = (
    part_filtered
    .merge(lineitem_filtered,
           left_on="P_PARTKEY", right_on="L_PARTKEY",
           how="inner")[["L_SUPPKEY", "L_ORDERKEY", "VOLUME"]]
)

# 4) join supplier
supplier_filtered = supplier[["S_SUPPKEY", "S_NATIONKEY"]]
total = total.merge(
    supplier_filtered,
    left_on="L_SUPPKEY", right_on="S_SUPPKEY",
    how="inner")[["L_ORDERKEY", "VOLUME", "S_NATIONKEY"]]

# 5) filter orders by date using string literals, extract year
orders_filtered = orders[
    (orders.O_ORDERDATE >= "1995-01-01") &
    (orders.O_ORDERDATE <  "1997-01-01")
]
orders_filtered["O_YEAR"] = orders_filtered.O_ORDERDATE.dt.year
orders_filtered = orders_filtered[["O_ORDERKEY", "O_CUSTKEY", "O_YEAR"]]

total = total.merge(
    orders_filtered,
    left_on="L_ORDERKEY", right_on="O_ORDERKEY",
    how="inner")[["VOLUME", "S_NATIONKEY", "O_CUSTKEY", "O_YEAR"]]

# 6) join customer
customer_filtered = customer[["C_CUSTKEY", "C_NATIONKEY"]]
total = total.merge(
    customer_filtered,
    left_on="O_CUSTKEY", right_on="C_CUSTKEY",
    how="inner")[["VOLUME", "S_NATIONKEY", "O_YEAR", "C_NATIONKEY"]]

# 7) join nation for customer→region and supplier→name
n1 = nation[["N_NATIONKEY", "N_REGIONKEY"]]
n2 = nation[["N_NATIONKEY", "N_NAME"]].rename(columns={"N_NAME": "NATION"})
total = total.merge(n1,
                     left_on="C_NATIONKEY", right_on="N_NATIONKEY",
                     how="inner")[["VOLUME", "S_NATIONKEY", "O_YEAR", "N_REGIONKEY"]]
total = total.merge(n2,
                     left_on="S_NATIONKEY", right_on="N_NATIONKEY",
                     how="inner")[["VOLUME", "O_YEAR", "N_REGIONKEY", "NATION"]]

# 8) filter region
region_filtered = region[region.R_NAME == "AMERICA"][["R_REGIONKEY"]]
total = total.merge(
    region_filtered,
    left_on="N_REGIONKEY", right_on="R_REGIONKEY",
    how="inner")[["VOLUME", "O_YEAR", "NATION"]]

# 9) compute market share without a Python UDF
# total volume per year
total_vol = total.groupby("O_YEAR").VOLUME.sum().reset_index(name="total_volume")
# brazil volume per year
brazil_vol = (
    total[total.NATION == "BRAZIL"]
    .groupby("O_YEAR").VOLUME
    .sum()
    .reset_index(name="brazil_volume")
)
# merge and compute share
total = total_vol.merge(brazil_vol, on="O_YEAR", how="left")
total["MKT_SHARE"] = total.brazil_volume / total.total_volume
# final projection and sort
total = total[["O_YEAR", "MKT_SHARE"]].sort_values("O_YEAR")

CPU times: user 112 ms, sys: 48.1 ms, total: 160 ms
Wall time: 169 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/rewritten/o4_mini_high/checkpoints/post_cell_7_try_4.pickle

migration speed (bps): 811723194.5012659
---------------------------
variables to migrate:
load_orders 144
part 48
orders 48
load_part 144
total 48
load_customer 144
customer 48
tpch_parent 54
STORAGE_OPTS 64
load_nation 144
nation 48
n2 48
orders_filtered 48
region_filtered 48
customer_filtered 48
region 48
part_filtered 48
load_region 144
DATA_ROOT 80
brazil_vol 48
n1 48
pd 72
supplier_filtered 48
total_vol 48
load_supplier 144
lineitem_filtered 48
supplier 48
lineitem 48
load_lineitem 144
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q08/rewritten/o4_mini_high/checkpoints/post_cell_7_try_4.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables ['part']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['part', 'lineitem']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['part', 'lineitem', 'supplier']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables ['customer']
Intermediate variables []
Future variables ['part', 'lineitem', 'supplier', 'orders']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['orders', 'supplier', 'part', 'lineitem', 'customer']
Modified dataframes
======= Cell 6 =======
Input varia

In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q08/opt_cell_exec_info_7_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[7], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q08/annotated/checkpoints/post_cell_7.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['lineitem']


me:  2
trying: ['load_lineitem']
me:  2
trying: ['load_orders']
me:  4
trying: ['part']
me:  1
trying: ['orders']


me:  4
trying: ['load_part']
me:  1
trying: ['region_filtered']
me:  8
trying: ['n2_filtered']
me:  8
trying: ['customer_filtered']
me:  8
trying: ['supplier_filtered']
me:  8
trying: ['part_filtered']
me:  8
trying: ['load_customer']
me:  5
trying: ['lineitem_filtered']


me:  8
trying: ['customer']
me:  5
trying: ['tpch_parent']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['load_nation']
me:  6
trying: ['nation']
me:  6
trying: ['total']
me:  8
trying: ['orders_filtered']
me:  8
trying: ['region']
me:  7
trying: ['n1_filtered']
me:  8
trying: ['orig_output']
me:  9
trying: ['load_region']
me:  7
trying: ['DATA_ROOT']
me:  0
trying: ['udf']
me:  8
trying: ['pd']
me:  0
trying: ['load_supplier']
me:  3
trying: ['supplier']
me:  3


Declaring variable tpch_parent
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable part
Declaring variable load_part
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable load_supplier
Declaring variable supplier
Declaring variable load_orders
Declaring variable orders
Declaring variable load_customer
Declaring variable customer
Declaring variable load_nation
Declaring variable nation
Declaring variable region
Declaring variable load_region
Declaring variable region_filtered
Declaring variable n2_filtered
Declaring variable customer_filtered
Declaring variable supplier_filtered
Declaring variable part_filtered
Declaring variable lineitem_filtered
Declaring variable total
Declaring variable orders_filtered
Declaring variable n1_filtered
Declaring variable udf
Declaring variable orig_output
